## 1. Carga de datos

Lectura e importación del fichero JSON en bruto para iniciar el proceso de aplanado (*flattening*) y estructuración de los datos.

In [16]:
import json
import pandas as pd
from IPython.display import display


pd.set_option('display.max_columns', None)

with open('user_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

for user in data:
    user.setdefault("userId", "unknown")       # asigna 'unknown' solo si no existe

df = pd.json_normalize(data).set_index('userId')
display(df.head())
display(df.shape)

,consumptions,consumptionSummaries,countryCode,city,yearOfBirth,gender,homeEnergyLabel,householdProfile
userId,,,,,,,,
c48be05f22d807690af96730960b5741b9e0cc2b66afecd331baf2c4cb906405,"[{'createdAt': {'_seconds': 1739660708, '_nano...",[],ES,Madrid,2005.0,female,None,None
459534a68d4f9828fdf2c7ab7346f7af93b3548dcca0737cbfd9b1d8942f17d5,[],[],EU,None,1995.0,female,None,None
bb082d002acd3fe83a7f3b04c14ce91a923d2e39926f2388ddda8f1ec10efb0b,[],[],ES,Madrid,2024.0,male,unsure,workersOrStudentsOutsideTheHome
b5135f676cb3a6c28d5c496d20a31eda5efd94f59307962a4a0daaa214be60c2,[],[],LV,None,2013.0,other,None,None
af553187df49982d43890043fb6dec88529276746f1ea814d1048b82dd3d9ec3,"[{'createdAt': {'_seconds': 1742166304, '_nano...",[],ES,None,2005.0,male,None,None


(1470, 8)

## 2. Aplanado de datos y exportación a CSV y XLSX

Aplanamos los datos del fichero JSON (habiendo analizado previamente su estructura estructurada) para su posterior conversión a los formatos CSV y XLSX.

In [ ]:
records = []

for user in data:
    # Campos comunes del usuario
    user_fields = {
        "userId": user.get("userId", "unknown"),
        "city": user.get("city"),
        "country": user.get("countryCode"),
        "gender": user.get("gender"),
        "yearOfBirth": user.get("yearOfBirth"),
        "homeEnergyLabel": user.get("homeEnergyLabel"),
        "householdProfile": user.get("householdProfile"),
    }

    for consumption in user.get("consumptions", []):
        # Mezclamos los campos del usuario con el diccionario del consumo (aún sin aplanar)
        combined = {**user_fields, **consumption}
        records.append(combined)

# json_normalize aplana automáticamente cualquier diccionario/lista anidado dentro de cada consumo
df = pd.json_normalize(records)

# Exportación
df.to_csv('consumptions.csv', index=False)
df.to_excel('consumptions.xlsx', index=False)
display(df.head())
display(df.shape)